# Issue #6: Segment all evaluation subsets with YAP (v2 — crash-safe)

Changes vs v1: a dead YAP server now ABORTS the segmenter instead of silently
writing raw text, failures are never cached, and the run cell auto-restarts
YAP and resumes until everything is segmented.

**Expected runtime:** 1–4 hours. Drive cache makes everything resumable.

In [ ]:
from getpass import getpass
token = getpass('GitHub PAT: ')
%cd /content
!rm -rf /content/NLP-Final-
!git clone https://{token}@github.com/AdonZahavi/NLP-Final-.git /content/NLP-Final-
%cd /content/NLP-Final-
!git checkout issue-6-segment-subsets
!git pull
!ls data/subsets/

In [ ]:
# Persist the segment cache on Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nlp_final_cache
!rm -rf /content/NLP-Final-/cache
!ln -s /content/drive/MyDrive/nlp_final_cache /content/NLP-Final-/cache

# ONE-TIME after the failed v1 run: the old cache contains raw-fallback
# garbage and MUST be deleted (only ~200 good texts are lost).
!rm -f /content/drive/MyDrive/nlp_final_cache/segment_cache.jsonl
!ls -la /content/NLP-Final-/cache/

In [ ]:
# Raw sentiment TSVs (needed by gold_check.py; data/raw is gitignored)
%cd /content/NLP-Final-
!mkdir -p data/raw
!wget -q -O data/raw/token_test.tsv https://github.com/omilab/Neural-Sentiment-Analyzer-for-Modern-Hebrew/raw/master/data/token_test.tsv
!wget -q -O data/raw/morph_test.tsv https://github.com/omilab/Neural-Sentiment-Analyzer-for-Modern-Hebrew/raw/master/data/morph_test.tsv
!wc -l data/raw/*.tsv

## Build YAP (same proven recipe)

In [ ]:
!apt-get update -qq && apt-get install -y -qq golang-go bzip2 > /dev/null 2>&1
import os
os.environ['GOPATH'] = '/content/gopath'
os.environ['GO111MODULE'] = 'off'

%cd /content
!rm -rf /content/gopath
!mkdir -p /content/gopath/src
!git clone -q https://github.com/OnlpLab/yap.git /content/gopath/src/yap
%cd /content/gopath/src/yap
!bunzip2 -k data/*.bz2
!git clone -q --depth 1 https://github.com/gorilla/mux.git vendor/github.com/gorilla/mux
!rm -rf vendor/github.com/gorilla/mux/.git
!mkdir -p vendor/gopkg.in
!git clone -q --depth 1 --branch v2 https://github.com/go-yaml/yaml.git vendor/gopkg.in/yaml.v2
!rm -rf vendor/gopkg.in/yaml.v2/.git
!ln -sf data/bgulex/bgupreflex_withdef.utf8.hr .
!ln -sf data/bgulex/bgulex.utf8.hr .
!go build -o /content/gopath/src/yap/yap_bin .
!ls -la /content/gopath/src/yap/yap_bin

In [ ]:
# YAP server manager: start (or restart) and wait until it answers
import subprocess, time, urllib.request, json

yap_proc = None

def start_yap(max_wait_rounds=60):
    global yap_proc
    if yap_proc is not None:
        try:
            yap_proc.kill(); yap_proc.wait()
        except Exception:
            pass
    logf = open('/content/yap.log', 'a')
    yap_proc = subprocess.Popen(
        ['./yap_bin', 'api'], cwd='/content/gopath/src/yap',
        stdout=logf, stderr=subprocess.STDOUT,
    )
    req = urllib.request.Request(
        'http://localhost:8000/yap/heb/joint',
        data=json.dumps({'text': 'שלום  '}).encode(),
        headers={'Content-Type': 'application/json'},
    )
    for attempt in range(max_wait_rounds):
        if yap_proc.poll() is not None:
            print(f'YAP EXITED (code {yap_proc.returncode}). Log tail:')
            print(open('/content/yap.log').read()[-2000:])
            return False
        try:
            with urllib.request.urlopen(req, timeout=120):
                print(f'YAP ready (attempt {attempt+1})')
                return True
        except Exception:
            time.sleep(10)
    print('YAP never became ready')
    return False

start_yap()

In [ ]:
# Smoke test: 3 sentiment records (must show 0 failed chunks, status [OK])
%cd /content/NLP-Final-
!python segmentation/segment_subsets.py --limit 3 --task sentiment
!head -c 600 data/subsets/segmented/sentiment_500.jsonl

In [ ]:
# FULL RUN with auto-recovery: if YAP dies, restart it and resume from cache.
import subprocess as sp

for round_no in range(1, 16):
    print(f'===== segmentation round {round_no} =====')
    ret = sp.call(['python', 'segmentation/segment_subsets.py'], cwd='/content/NLP-Final-')
    if ret == 0:
        print('SEGMENTATION COMPLETE ✔')
        break
    print(f'run exited with code {ret} — restarting YAP and resuming...')
    if not start_yap():
        print('could not restart YAP; check the log above')
        break
else:
    print('gave up after 15 rounds — inspect /content/yap.log')

## Validate

In [ ]:
# Hard check: zero empty _seg fields allowed
import json
for f in ['sentiment_500', 'nli_884', 'qa_500']:
    recs = [json.loads(l) for l in open(f'data/subsets/segmented/{f}.jsonl', encoding='utf-8')]
    empty = sum(1 for r in recs for k, v in r.items() if k.endswith('_seg') and not v)
    print(f'{f}: {len(recs)} records, {empty} empty seg fields', '✔' if empty == 0 else '✘ RE-RUN')

In [ ]:
!pip install -q pandas
%cd /content/NLP-Final-
!python segmentation/gold_check.py

In [ ]:
!head -60 data/subsets/segmented/qc_sample.md

## Commit results (overwrites the bad v1 files)

In [ ]:
%cd /content/NLP-Final-
!git config user.email "orna.zahavi1@gmail.com" && git config user.name "Or Zahavi"
!git add data/subsets/segmented/ segmentation/segment_subsets.py
!git commit -m "Issue #6 v2: complete YAP-segmented subsets (crash-safe rerun)"
!git push origin issue-6-segment-subsets